# 3 — Prepare Model Data

This is the **final preprocessing stage** before training-example generation and model training.

### Input
`conditions_symptom_parsed_with_manual_overrides.json`

### What this notebook does
- separates trusted records from records that still need review
- creates a canonical symptom representation
- creates model-specific text for TF-IDF / Logistic Regression, RNN / LSTM, and BERT / BioBERT
- tokenizes the classical/RNN text
- supports **optional** lemmatization
- validates the final preprocessing result
- saves model-ready JSON/CSV files

### Important
Only `symptom_phrases` are used as diagnosis-model input here.  
`causes`, `warnings`, and `recommendations` are preserved for later retrieval/explanation, but are **not** mixed into the classifier input.


## 1. Imports and project paths

In [51]:
from pathlib import Path
from collections import Counter
import csv
import json
import re
import unicodedata


def find_project_root() -> Path:
    """Find the project folder that contains app/."""
    current = Path.cwd().resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "app").is_dir():
            return candidate

    # Fallback if the notebook is being tested outside the repo.
    return current


PROJECT_ROOT = find_project_root()

DATASET_DIR = PROJECT_ROOT / "app" / "dataset"
INPUT_PATH = (
    DATASET_DIR
    / "output"
    / "preprocessing"
    / "conditions_symptom_parsed_with_manual_overrides.json"
)

OUTPUT_DIR = (
    DATASET_DIR
    / "output"
    / "preprocessing"
)

# Helpful fallback when testing the notebook outside the repo.
if not INPUT_PATH.exists():
    fallback = PROJECT_ROOT / "conditions_symptom_parsed_with_manual_overrides.json"
    if fallback.exists():
        INPUT_PATH = fallback

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_READY_JSON = OUTPUT_DIR / "conditions_model_ready.json"
MODEL_READY_CSV = OUTPUT_DIR / "conditions_model_ready.csv"
REVIEW_JSON = OUTPUT_DIR / "conditions_model_review.json"

print("Project root:", PROJECT_ROOT)
print("Input:", INPUT_PATH)
print("Output directory:", OUTPUT_DIR)


Project root: C:\Users\po\Downloads\medical-diagnosis
Input: C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\preprocessing\conditions_symptom_parsed_with_manual_overrides.json
Output directory: C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\preprocessing


## 2. Load the final symptom-parsed dataset

In [52]:
if not INPUT_PATH.exists():
    raise FileNotFoundError(
        f"Could not find input dataset: {INPUT_PATH}\n"
        "Place conditions_symptom_parsed_with_manual_overrides.json under "
        "app/dataset/output/preprocessing/."
    )

with INPUT_PATH.open("r", encoding="utf-8") as f:
    conditions = json.load(f)

if not isinstance(conditions, list):
    raise TypeError("Expected the input JSON to contain a list of condition records.")

print("Loaded conditions:", len(conditions))


Loaded conditions: 406


## 3. Validate the incoming parser output

In [53]:
REQUIRED_FIELDS = {
    "condition",
    "symptom_phrases",
    "symptom_phrase_count",
    "parser_method",
    "parser_needs_review",
}

missing_field_records = []

for index, record in enumerate(conditions):
    missing = REQUIRED_FIELDS - set(record.keys())
    if missing:
        missing_field_records.append(
            {
                "index": index,
                "condition": record.get("condition"),
                "missing": sorted(missing),
            }
        )

if missing_field_records:
    raise ValueError(
        "Some records are missing required parser fields:\n"
        + json.dumps(missing_field_records[:10], indent=2, ensure_ascii=False)
    )

condition_names = [
    str(record.get("condition", "")).strip()
    for record in conditions
]

duplicate_conditions = [
    name
    for name, count in Counter(condition_names).items()
    if name and count > 1
]

empty_condition_names = sum(not name for name in condition_names)

print("Empty condition names:", empty_condition_names)
print("Duplicate condition names:", len(duplicate_conditions))
print("With symptom phrases:", sum(r["symptom_phrase_count"] > 0 for r in conditions))
print("Zero symptom phrases:", sum(r["symptom_phrase_count"] == 0 for r in conditions))
print("Needs review:", sum(bool(r["parser_needs_review"]) for r in conditions))
print("Parser methods:", dict(Counter(r["parser_method"] for r in conditions)))

assert empty_condition_names == 0, "Condition names must not be empty."
assert not duplicate_conditions, f"Duplicate conditions found: {duplicate_conditions[:10]}"


Empty condition names: 0
Duplicate condition names: 0
With symptom phrases: 390
Zero symptom phrases: 16
Needs review: 32
Parser methods: {'structured': 273, 'manual_override': 60, 'plain_list_fallback': 48, 'none': 16, 'narrative_fallback': 9}


## 4. Split trusted records from records that still need review

A record is trusted for model preparation only when:
- it has at least one symptom phrase, and
- `parser_needs_review` is `False`

Anything else is held out of training for now.


In [54]:
trusted_conditions = [
    record
    for record in conditions
    if (
        record["symptom_phrase_count"] > 0
        and not record["parser_needs_review"]
    )
]

review_conditions = [
    record
    for record in conditions
    if record not in trusted_conditions
]

print("Trusted for model preparation:", len(trusted_conditions))
print("Held for review:", len(review_conditions))
print("Total:", len(trusted_conditions) + len(review_conditions))

assert len(trusted_conditions) + len(review_conditions) == len(conditions)


Trusted for model preparation: 374
Held for review: 32
Total: 406


## 5. Text normalization and tokenization

We keep this conservative.

- Multi-word medical phrases remain intact in `symptoms_canonical`.
- TF-IDF / Logistic Regression receives a lowercase, lightly normalized string.
- RNN / LSTM receives tokenized text.
- BERT / BioBERT receives the readable canonical text because its pretrained tokenizer should do the real tokenization later.
- We do **not** remove stopwords here because words such as `no`, `not`, `without`, and phrase structure can be medically meaningful.


In [55]:
TOKEN_RE = re.compile(
    r"[^\W_]+(?:['’-][^\W_]+)*|\d+(?:\.\d+)?",
    flags=re.UNICODE,
)


def normalize_space(text: str) -> str:
    text = unicodedata.normalize("NFC", str(text or ""))
    text = re.sub(r"\s+", " ", text)
    return text.strip()


def normalize_classical(text: str) -> str:
    """Light normalization for classical models / vocabulary building."""
    text = normalize_space(text).lower()

    # Keep apostrophes and hyphens inside words where possible,
    # but turn other punctuation into spaces.
    text = re.sub(
        r"[^\w\s'’-]",
        " ",
        text,
        flags=re.UNICODE,
    )
    text = re.sub(r"\s+", " ", text).strip()

    return text


def tokenize_text(text: str) -> list[str]:
    text = normalize_classical(text)
    return TOKEN_RE.findall(text)


def build_canonical_symptoms(symptom_phrases: list[str]) -> str:
    cleaned = [
        normalize_space(phrase)
        for phrase in symptom_phrases
        if normalize_space(phrase)
    ]
    return " ; ".join(cleaned)


## 6. Optional lemmatization

Lemmatization is **off by default** because:
- it is not needed for BERT/BioBERT,
- TF-IDF often performs well without it,
- aggressive normalization can damage medical wording.

If you want to compare a lemmatized classical/RNN version later, set `USE_LEMMATIZATION = True`.

That optional path uses NLTK WordNet. If enabled, install NLTK and make sure the `wordnet` corpus is available.


after comparisions its decided to not apply lemmatization



In [56]:
USE_LEMMATIZATION = False


def get_lemmatizer():
    if not USE_LEMMATIZATION:
        return None

    try:
        from nltk.stem import WordNetLemmatizer
        from nltk.corpus import wordnet
    except ImportError as exc:
        raise ImportError(
            "Lemmatization is enabled but NLTK is not installed. "
            "Install it with: pip install nltk"
        ) from exc

    # Check that WordNet data is available without downloading silently.
    try:
        wordnet.ensure_loaded()
    except LookupError as exc:
        raise LookupError(
            "NLTK WordNet data is missing. Run this once in Python:\n"
            "import nltk\n"
            "nltk.download('wordnet')\n"
            "nltk.download('omw-1.4')"
        ) from exc

    return WordNetLemmatizer()


LEMMATIZER = get_lemmatizer()


def lemmatize_tokens(tokens: list[str]) -> list[str]:
    if LEMMATIZER is None:
        return tokens

    # Conservative noun-default lemmatization.
    # We keep this optional rather than pretending POS tags are known.
    return [
        LEMMATIZER.lemmatize(token)
        for token in tokens
    ]


## 7. Build model-ready records

In [57]:
model_ready = []

for record in trusted_conditions:
    prepared = dict(record)

    symptom_phrases = [
        normalize_space(phrase)
        for phrase in record["symptom_phrases"]
        if normalize_space(phrase)
    ]

    symptoms_canonical = build_canonical_symptoms(symptom_phrases)
    symptoms_classical = normalize_classical(symptoms_canonical)
    tokens = tokenize_text(symptoms_canonical)
    model_tokens = lemmatize_tokens(tokens)

    prepared["symptom_phrases"] = symptom_phrases

    # Standard readable representation.
    prepared["symptoms_canonical"] = symptoms_canonical

    # TF-IDF / Logistic Regression input.
    prepared["symptoms_classical"] = " ".join(model_tokens)

    # RNN / LSTM preprocessing input.
    # Integer vocabulary IDs and padding happen during the model pipeline.
    prepared["symptoms_tokens"] = model_tokens

    # BERT / BioBERT input.
    # Do not manually lemmatize or aggressively tokenize this version.
    prepared["symptoms_bert"] = symptoms_canonical

    prepared["lemmatization_used"] = USE_LEMMATIZATION

    model_ready.append(prepared)

print("Model-ready records:", len(model_ready))


Model-ready records: 374


## 8. Final validation

These checks protect the training stage from accidentally receiving empty, duplicate, or unresolved records.


In [58]:
empty_canonical = [
    r["condition"]
    for r in model_ready
    if not r["symptoms_canonical"].strip()
]

empty_classical = [
    r["condition"]
    for r in model_ready
    if not r["symptoms_classical"].strip()
]

empty_tokens = [
    r["condition"]
    for r in model_ready
    if not r["symptoms_tokens"]
]

review_leakage = [
    r["condition"]
    for r in model_ready
    if r["parser_needs_review"]
]

duplicate_model_conditions = [
    name
    for name, count in Counter(
        r["condition"] for r in model_ready
    ).items()
    if count > 1
]

print("Final preprocessing validation")
print("------------------------------")
print("Model-ready records:", len(model_ready))
print("Empty canonical text:", len(empty_canonical))
print("Empty classical text:", len(empty_classical))
print("Empty token lists:", len(empty_tokens))
print("Review records leaked into model-ready set:", len(review_leakage))
print("Duplicate model conditions:", len(duplicate_model_conditions))

assert not empty_canonical
assert not empty_classical
assert not empty_tokens
assert not review_leakage
assert not duplicate_model_conditions

print("\nValidation passed ✅")


Final preprocessing validation
------------------------------
Model-ready records: 374
Empty canonical text: 0
Empty classical text: 0
Empty token lists: 0
Review records leaked into model-ready set: 0
Duplicate model conditions: 0

Validation passed ✅


## 9. Preview a few prepared records

In [59]:
PREVIEW_COUNT = 5

for record in model_ready[:PREVIEW_COUNT]:
    print("=" * 80)
    print("Condition:", record["condition"])
    print("Parser method:", record["parser_method"])
    print("Canonical:", record["symptoms_canonical"])
    print("Classical:", record["symptoms_classical"])
    print("Tokens:", record["symptoms_tokens"][:30])
    print("BERT:", record["symptoms_bert"])
    print()


Condition: Abdominal aortic aneurysm
Parser method: structured
Canonical: pain or a pulsating feeling in the abdomen, or persistent back pain
Classical: pain or a pulsating feeling in the abdomen or persistent back pain
Tokens: ['pain', 'or', 'a', 'pulsating', 'feeling', 'in', 'the', 'abdomen', 'or', 'persistent', 'back', 'pain']
BERT: pain or a pulsating feeling in the abdomen, or persistent back pain

Condition: Aplastic anaemia
Parser method: structured
Canonical: fatigue ; shortness of breath ; headaches ; chest pains ; bleeding ; bruising ; reoccurring severe infections ; brain fog ; poor concentration
Classical: fatigue shortness of breath headaches chest pains bleeding bruising reoccurring severe infections brain fog poor concentration
Tokens: ['fatigue', 'shortness', 'of', 'breath', 'headaches', 'chest', 'pains', 'bleeding', 'bruising', 'reoccurring', 'severe', 'infections', 'brain', 'fog', 'poor', 'concentration']
BERT: fatigue ; shortness of breath ; headaches ; chest pains ;

## 10. Save the final preprocessing outputs

`conditions_model_ready.json` is the **final preprocessing dataset** that should feed the next stage.

`conditions_model_review.json` keeps unresolved records separate so they are not silently used in training.

The CSV is included for easy inspection in Excel / VS Code.


In [60]:
SAVE_OUTPUTS = True

if SAVE_OUTPUTS:
    with MODEL_READY_JSON.open("w", encoding="utf-8") as f:
        json.dump(
            model_ready,
            f,
            ensure_ascii=False,
            indent=2,
        )

    # Save a human-friendly flat CSV.
    csv_fields = [
        "condition",
        "symptoms_canonical",
        "symptoms_classical",
        "symptoms_tokens",
        "symptoms_bert",
        "parser_method",
        "lemmatization_used",
    ]

    with MODEL_READY_CSV.open("w", encoding="utf-8-sig", newline="") as f:
        writer = csv.DictWriter(
            f,
            fieldnames=csv_fields,
        )
        writer.writeheader()

        for record in model_ready:
            writer.writerow(
                {
                    "condition": record["condition"],
                    "symptoms_canonical": record["symptoms_canonical"],
                    "symptoms_classical": record["symptoms_classical"],
                    "symptoms_tokens": " | ".join(record["symptoms_tokens"]),
                    "symptoms_bert": record["symptoms_bert"],
                    "parser_method": record["parser_method"],
                    "lemmatization_used": record["lemmatization_used"],
                }
            )

    review_output = []

    for record in review_conditions:
        review_record = dict(record)

        if record.get("symptom_phrase_count", 0) == 0:
            review_reason = "zero_symptom_phrases"
        elif record.get("parser_needs_review"):
            review_reason = "parser_needs_review"
        else:
            review_reason = "not_trusted"

        review_record["model_prep_review_reason"] = review_reason
        review_output.append(review_record)

    with REVIEW_JSON.open("w", encoding="utf-8") as f:
        json.dump(
            review_output,
            f,
            ensure_ascii=False,
            indent=2,
        )

    print("Saved:")
    print("-", MODEL_READY_JSON)
    print("-", MODEL_READY_CSV)
    print("-", REVIEW_JSON)
else:
    print("SAVE_OUTPUTS=False — nothing was written.")


Saved:
- C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\preprocessing\conditions_model_ready.json
- C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\preprocessing\conditions_model_ready.csv
- C:\Users\po\Downloads\medical-diagnosis\app\dataset\output\preprocessing\conditions_model_review.json


## Preprocessing complete ✅

After this notebook, the next stage is **training-example generation**, not more preprocessing.

Pipeline:

```text
conditions_model_ready.json
        ↓
generate source-grounded symptom combinations
        ↓
training examples: X = symptom text, y = condition
        ↓
TF-IDF + Logistic Regression baseline
        ↓
RNN / LSTM
        ↓
BERT / BioBERT
```

Do not generate synthetic medical facts or synonyms during training-example generation.  
Examples should be constructed only from the reviewed symptom phrases already present in this dataset.
